> **Chapter Map:** This notebook is the companion for **Chapter 3** (Reactor Design and Catalyst Characterization).

# NB3: Reactor Design and Catalyst Characterization

## Learning Objectives

After completing this notebook, you will be able to:

1. Solve the batch reactor ODE for first-order and Langmuir-Hinshelwood kinetics
2. Apply the CSTR design equation to calculate exit conversion for simple and surface kinetics
3. Integrate the PFR design equation using `solve_ivp`
4. **Compare differential and integral methods** for determining reaction order (LO4)
5. Compare CSTR and PFR performance for different reaction orders
6. **Predict reactor conversion across temperatures** using Arrhenius $k(T)$ (LO5)
7. Calculate turnover frequency (TOF) and turnover number (TON) from conversion data
8. Convert between space time, space velocity (WHSV, GHSV, LHSV), and contact time
9. Perform BET surface area analysis from nitrogen adsorption data
10. Apply the Redhead equation to extract desorption energies from TPD spectra
11. Use the Scherrer equation to estimate crystallite size from XRD peak widths
12. Calculate metal dispersion and active surface area from chemisorption data
13. Combine multiple characterization techniques into a complete catalyst profile

## Background

This notebook has two parts that together form the core of Chapter 3.

**Part I (Reactor Design)** translates the rate expressions developed in Chapter 2 into engineering practice. The choice of reactor type -- batch, CSTR, or PFR -- profoundly affects conversion, selectivity, and economics. In heterogeneous catalysis, the design equations must account for surface kinetics (Langmuir-Hinshelwood) rather than simple power-law models.

**Part II (Catalyst Characterization)** answers the question: *how do we know the catalyst's surface area, binding energies, and crystallite structure?* The key experimental techniques -- BET, TPD, XRD, and chemisorption -- provide the physical parameters that feed into reactor design and kinetic modeling.

### Key Equations

**Batch Reactor (constant volume):**
$$-\frac{dC_A}{dt} = r(C_A)$$

**CSTR Design Equation:**
$$\tau = \frac{C_{A0} \cdot X}{r(C_{A,\text{exit}})}$$

**PFR Design Equation:**
$$\frac{dX}{d\tau} = \frac{r(C_A)}{C_{A0}}, \quad C_A = C_{A0}(1-X)$$

**Turnover Frequency:**
$$\text{TOF} = \frac{\text{moles of product}}{\text{moles of active sites} \times \text{time}}$$

**BET Equation:**
$$\frac{P/P_0}{V(1-P/P_0)} = \frac{1}{V_m c} + \frac{c-1}{V_m c}\left(\frac{P}{P_0}\right)$$

**Redhead Equation:**
$$E_d = RT_p\left[\ln\left(\frac{\nu T_p}{\beta}\right) - 3.64\right]$$

**Scherrer Equation:**
$$D = \frac{K\lambda}{\beta\cos\theta}$$

## Prerequisites

- **Concepts:** Langmuir isotherm, LH/ER/MVK rate laws, Arrhenius equation (Chapter 2)
- **Python skills:** numpy arrays, matplotlib, scipy.optimize, scipy.integrate (NB1, NB2)
- **Previous notebooks:** NB1 (Bridge), NB2 (Surface Kinetics & Temperature Dependence)

---
## Setup

Run the cell below to import libraries and set plot defaults. Functions for reactor design, TOF/TON, and characterization analysis are defined in later cells.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import fsolve, curve_fit
from scipy.integrate import solve_ivp
from scipy import stats

# Set default plot style for publication-quality figures
plt.rcParams['figure.figsize'] = (8, 6)
plt.rcParams['figure.dpi'] = 150
plt.rcParams['figure.constrained_layout.use'] = False
plt.rcParams['font.size'] = 12
plt.rcParams['axes.labelsize'] = 14
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['legend.fontsize'] = 11
plt.rcParams['xtick.labelsize'] = 11
plt.rcParams['ytick.labelsize'] = 11
plt.rcParams['lines.linewidth'] = 2
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

# Colorblind-safe color palette (Wong, 2011)
COLORS = {
    'blue': '#0072B2',
    'orange': '#E69F00',
    'green': '#009E73',
    'vermillion': '#D55E00',
    'skyblue': '#56B4E9',
    'purple': '#CC79A7'
}

# Physical constants
R = 8.314          # Gas constant, J/(mol*K)
N_A = 6.022e23     # Avogadro's number, mol^-1
V_mol_STP = 22414  # Molar volume at STP, cm^3/mol

print("Setup complete. Libraries imported successfully.")

---
## Part 1: Batch Reactor Kinetics

The batch reactor is a closed vessel where concentration changes with time. The fundamental mass balance for constant volume is:

$$-\frac{dC_A}{dt} = r(C_A)$$

For simple power-law kinetics $r = kC_A^n$, analytical solutions exist:

- **Zeroth-order** ($n=0$): $C_A = C_{A0} - kt$
- **First-order** ($n=1$): $C_A = C_{A0}\exp(-kt)$
- **Second-order** ($n=2$): $1/C_A = 1/C_{A0} + kt$

For more complex rate laws (e.g., Langmuir-Hinshelwood), we must solve the ODE numerically.

### 1.1 Analytical Solutions for Power-Law Kinetics

In [ ]:
def batch_analytical(t, k, C_A0, order=1):
    """
    Analytical solution for batch reactor with power-law kinetics.

    Parameters
    ----------
    t : float or array-like
        Reaction time (s)
    k : float
        Rate constant (units depend on order)
    C_A0 : float
        Initial concentration (mol/L)
    order : int
        Reaction order (0, 1, or 2)

    Returns
    -------
    ndarray
        Concentration C_A(t) in mol/L

    Notes
    -----
    For zeroth-order, concentration is clipped at zero
    (reaction stops when reactant is consumed).
    """
    t = np.asarray(t, dtype=float)

    if order == 0:
        C_A = C_A0 - k * t
        return np.maximum(C_A, 0.0)  # Cannot go negative
    elif order == 1:
        return C_A0 * np.exp(-k * t)
    elif order == 2:
        return C_A0 / (1 + k * C_A0 * t)
    else:
        raise ValueError(f"Order {order} not implemented analytically.")


# Reactor parameters
C_A0 = 1.0    # mol/L
k_first = 0.05  # s^-1

# Time array
t = np.linspace(0, 100, 500)  # seconds

# Compare all three orders (rate constants chosen for comparable timescales)
k_zeroth = 0.01     # mol/(L*s)
k_second = 0.10     # L/(mol*s)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

orders = [0, 1, 2]
k_values = [k_zeroth, k_first, k_second]
colors = [COLORS['blue'], COLORS['orange'], COLORS['green']]
labels = ['Zeroth-order', 'First-order', 'Second-order']
linestyles = ['-', '--', '-.']

for order, k_val, color, label, ls in zip(orders, k_values, colors, labels, linestyles):
    C_A = batch_analytical(t, k_val, C_A0, order=order)
    X = 1 - C_A / C_A0  # Conversion

    ax1.plot(t, C_A, color=color, linestyle=ls, label=label)
    ax2.plot(t, X, color=color, linestyle=ls, label=label)

# Left panel: concentration
ax1.set_xlabel('Time (s)')
ax1.set_ylabel('Concentration, $C_A$ (mol/L)')
ax1.set_title('Batch Reactor: $C_A(t)$')
ax1.legend(loc='upper right')
ax1.set_ylim(0, 1.1)

# Right panel: conversion
ax2.set_xlabel('Time (s)')
ax2.set_ylabel('Conversion, $X$')
ax2.set_title('Batch Reactor: $X(t)$')
ax2.legend(loc='lower right')
ax2.set_ylim(0, 1.05)

plt.tight_layout()
plt.show()

# Print half-lives for comparison
t_half_0 = C_A0 / (2 * k_zeroth)
t_half_1 = np.log(2) / k_first
t_half_2 = 1 / (k_second * C_A0)
print("Half-Life Comparison:")
print(f"  Zeroth-order: t_1/2 = {t_half_0:.1f} s (depends on C_A0)")
print(f"  First-order:  t_1/2 = {t_half_1:.1f} s (independent of C_A0)")
print(f"  Second-order: t_1/2 = {t_half_2:.1f} s (depends on C_A0)")

### 1.2 Numerical Solution for Langmuir-Hinshelwood Kinetics

In heterogeneous catalysis, the rate often follows Langmuir-Hinshelwood kinetics rather than simple power-law. For a unimolecular surface reaction:

$$r = \frac{k \cdot K \cdot C_A}{1 + K \cdot C_A}$$

This cannot be integrated analytically in general, so we use `scipy.integrate.solve_ivp`.

In [ ]:
def batch_lh_ode(t, C_A, k_s, K):
    """
    ODE for batch reactor with unimolecular LH kinetics.

    Parameters
    ----------
    t : float
        Time (s) -- required by solve_ivp but not used explicitly
    C_A : float
        Concentration of A (mol/L)
    k_s : float
        Surface rate constant (mol/(L*s))
    K : float
        Adsorption equilibrium constant (L/mol)

    Returns
    -------
    float
        dC_A/dt

    Notes
    -----
    Rate law: r = k_s * K * C_A / (1 + K * C_A)
    dC_A/dt = -r
    """
    r = k_s * K * C_A / (1 + K * C_A)
    return -r


# Parameters
C_A0 = 1.0     # mol/L
k_s = 0.05     # mol/(L*s), maximum rate when surface is saturated
K_ads = 5.0    # L/mol, adsorption equilibrium constant

# Solve the ODE
t_span = (0, 200)  # seconds
t_eval = np.linspace(0, 200, 500)

sol_lh = solve_ivp(
    batch_lh_ode,
    t_span,
    [C_A0],
    args=(k_s, K_ads),
    t_eval=t_eval,
    method='RK45',
    rtol=1e-10,
    atol=1e-12
)

C_A_lh = sol_lh.y[0]
X_lh = 1 - C_A_lh / C_A0

# Compare LH kinetics with first-order kinetics (same initial rate)
# Initial LH rate: r0 = k_s * K * C_A0 / (1 + K * C_A0)
r0_lh = k_s * K_ads * C_A0 / (1 + K_ads * C_A0)
k_equiv = r0_lh / C_A0  # Equivalent first-order k giving same initial rate
C_A_first = batch_analytical(t_eval, k_equiv, C_A0, order=1)
X_first = 1 - C_A_first / C_A0

# Plot comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

ax1.plot(t_eval, C_A_lh, '-', color=COLORS['blue'], linewidth=2.5,
         label=f'LH ($k_s$={k_s}, $K$={K_ads})')
ax1.plot(t_eval, C_A_first, '--', color=COLORS['vermillion'], linewidth=2,
         label=f'First-order ($k$={k_equiv:.4f})')
ax1.set_xlabel('Time (s)')
ax1.set_ylabel('Concentration, $C_A$ (mol/L)')
ax1.set_title('Batch Reactor: LH vs First-Order')
ax1.legend(loc='upper right')

ax2.plot(t_eval, X_lh, '-', color=COLORS['blue'], linewidth=2.5,
         label='LH kinetics')
ax2.plot(t_eval, X_first, '--', color=COLORS['vermillion'], linewidth=2,
         label='First-order')
ax2.set_xlabel('Time (s)')
ax2.set_ylabel('Conversion, $X$')
ax2.set_title('Conversion Profiles')
ax2.legend(loc='lower right')
ax2.set_ylim(0, 1.05)

plt.tight_layout()
plt.show()

print(f"Initial LH rate: r0 = {r0_lh:.4f} mol/(L*s)")
print(f"Equivalent first-order k = {k_equiv:.4f} s^-1")
print(f"\nAt t = 100 s:")
print(f"  LH conversion:         X = {np.interp(100, t_eval, X_lh):.3f}")
print(f"  First-order conversion: X = {np.interp(100, t_eval, X_first):.3f}")
print("\nNote: LH kinetics transitions from first-order (low C_A, K*C_A << 1)")
print("to zeroth-order (high C_A, K*C_A >> 1) behavior.")

### Key Observations

1. **At early times** (high $C_A$, surface saturated): LH rate approaches the maximum $k_s$ (zeroth-order regime)
2. **At late times** (low $C_A$, surface mostly vacant): LH rate becomes proportional to $C_A$ (first-order regime)
3. The first-order model initially matches LH but diverges because it does not capture the saturation effect

### ✏️ Concept Check: Batch Reactor Kinetics

**Before continuing, predict:**

1. A batch reactor with LH kinetics ($k_s = 0.1$ mol/(L·s), $K = 5$ L/mol) starts with $C_{A0} = 2.0$ mol/L. At early times, is the effective reaction order closer to zeroth or first order? Why?

2. If you double $C_{A0}$ in a first-order batch reactor, does the half-life change? What about a second-order reactor?

<details>
<summary>Click to reveal answers</summary>

1. **Zeroth order.** At high $C_A$, $KC_A \gg 1$, so $r \approx k_s$ (rate is constant, independent of $C_A$). Here $KC_{A0} = 5 \times 2 = 10 \gg 1$, confirming zeroth-order behavior initially.

2. **First order:** $t_{1/2} = \ln 2/k$ — independent of $C_{A0}$. **Second order:** $t_{1/2} = 1/(kC_{A0})$ — halves when $C_{A0}$ doubles. This is a key diagnostic for reaction order.
</details>

---
## Part 2: CSTR Design

The CSTR (Continuous Stirred Tank Reactor) operates at steady state with perfect mixing. The exit concentration equals the concentration everywhere inside the reactor.

**Design equation:**
$$\tau = \frac{V}{\dot{V}} = \frac{C_{A0} \cdot X}{r(C_{A,\text{exit}})}$$

where $\tau$ is the space time, $C_{A,\text{exit}} = C_{A0}(1-X)$.

For first-order kinetics $r = kC_A$, the analytical solution is:
$$X = \frac{k\tau}{1 + k\tau}$$

In [ ]:
def cstr_conversion_analytical(tau, k, C_A0, order=1):
    """
    Analytical CSTR conversion for power-law kinetics.

    Parameters
    ----------
    tau : float or array-like
        Space time (s)
    k : float
        Rate constant (units depend on order)
    C_A0 : float
        Inlet concentration (mol/L)
    order : int
        Reaction order (1 or 2)

    Returns
    -------
    ndarray
        Conversion X (dimensionless, 0 to 1)

    Notes
    -----
    First-order:  X = k*tau / (1 + k*tau)
    Second-order: Solve quadratic from tau = C_A0*X / (k*(C_A0*(1-X))^2)
    """
    tau = np.asarray(tau, dtype=float)

    if order == 1:
        return k * tau / (1 + k * tau)
    elif order == 2:
        # CSTR balance: tau = C_A0*X / (k*C_A0^2*(1-X)^2)
        # Let Da = k*C_A0*tau (Damkohler number)
        Da = k * C_A0 * tau
        # Da*(1-X)^2 = X => Da*X^2 - (2*Da+1)*X + Da = 0
        discriminant = (2 * Da + 1)**2 - 4 * Da**2
        X = ((2 * Da + 1) - np.sqrt(discriminant)) / (2 * Da)
        # Handle Da=0 case
        X = np.where(Da < 1e-15, 0.0, X)
        return X
    else:
        raise ValueError(f"Order {order} not implemented.")


# Demonstrate first-order CSTR
k = 0.05  # s^-1
C_A0 = 1.0  # mol/L
tau_range = np.linspace(0, 200, 500)

X_cstr = cstr_conversion_analytical(tau_range, k, C_A0, order=1)

fig, ax = plt.subplots(figsize=(10, 7))

ax.plot(tau_range, X_cstr, '-', color=COLORS['blue'], linewidth=2.5,
        label='CSTR ($X = k\\tau/(1+k\\tau)$)')

# Mark key conversion levels
for X_target, ls in zip([0.5, 0.9, 0.95], ['--', '-.', ':']):
    tau_target = X_target / (k * (1 - X_target))  # Rearranged CSTR eq.
    ax.axhline(y=X_target, color='gray', linestyle=ls, alpha=0.5)
    ax.plot(tau_target, X_target, 'o', color=COLORS['blue'], markersize=10,
            markeredgecolor='black', markeredgewidth=1.5)
    ax.annotate(f'$X$ = {X_target}, $\\tau$ = {tau_target:.0f} s',
                xy=(tau_target, X_target),
                xytext=(tau_target + 10, X_target - 0.06),
                fontsize=10)

ax.set_xlabel('Space Time, $\\tau$ (s)')
ax.set_ylabel('Conversion, $X$')
ax.set_title(f'CSTR Performance: First-Order Kinetics ($k$ = {k} s$^{{-1}}$)')
ax.set_xlim(0, 200)
ax.set_ylim(0, 1.05)
ax.legend(loc='lower right', fontsize=12)

plt.tight_layout()
plt.show()

print("CSTR Space Time Requirements (First-Order, k = 0.05 s^-1):")
print("-" * 50)
for X_target in [0.50, 0.80, 0.90, 0.95, 0.99]:
    tau_req = X_target / (k * (1 - X_target))
    print(f"  X = {X_target:.2f}: tau = {tau_req:>8.1f} s")
print("\nNote: The cost of the last 10% conversion is enormous!")

### 2.1 CSTR with Langmuir-Hinshelwood Kinetics

For LH kinetics in a CSTR, the design equation becomes implicit (cannot be solved algebraically for $X$):

$$\tau = \frac{C_{A0} X}{k_s K C_{A0}(1-X)/(1 + K C_{A0}(1-X))}$$

We must use `fsolve` to find the exit conversion numerically.

In [ ]:
def cstr_lh_residual(X, tau, k_s, K, C_A0):
    """
    Residual function for CSTR with unimolecular LH kinetics.

    The CSTR design equation is:
        tau = C_A0 * X / r(C_A_exit)
    Rearranged to residual form:
        f(X) = C_A0 * X - tau * r(C_A_exit) = 0

    Parameters
    ----------
    X : float
        Conversion (unknown to solve for)
    tau : float
        Space time (s)
    k_s : float
        Surface rate constant (mol/(L*s))
    K : float
        Adsorption equilibrium constant (L/mol)
    C_A0 : float
        Inlet concentration (mol/L)

    Returns
    -------
    float
        Residual (should be zero at solution)
    """
    C_A_exit = C_A0 * (1 - X)
    r_exit = k_s * K * C_A_exit / (1 + K * C_A_exit)
    return C_A0 * X - tau * r_exit


# Parameters
k_s = 0.05     # mol/(L*s)
K_ads = 5.0    # L/mol
C_A0 = 1.0     # mol/L

# Solve for conversion at each space time
tau_range = np.linspace(0.1, 300, 500)
X_cstr_lh = np.zeros_like(tau_range)

for i, tau_val in enumerate(tau_range):
    X_guess = 0.5
    X_sol = fsolve(cstr_lh_residual, X_guess,
                   args=(tau_val, k_s, K_ads, C_A0),
                   full_output=False)
    X_cstr_lh[i] = np.clip(X_sol[0], 0, 1)

# Compare with equivalent first-order CSTR
# Match initial rate: r0 = k_s*K*C_A0/(1+K*C_A0)
r0 = k_s * K_ads * C_A0 / (1 + K_ads * C_A0)
k_equiv = r0 / C_A0
X_cstr_first = cstr_conversion_analytical(tau_range, k_equiv, C_A0, order=1)

# Plot comparison
fig, ax = plt.subplots(figsize=(10, 7))

ax.plot(tau_range, X_cstr_lh, '-', color=COLORS['blue'], linewidth=2.5,
        label=f'LH kinetics ($k_s$={k_s}, $K$={K_ads})')
ax.plot(tau_range, X_cstr_first, '--', color=COLORS['vermillion'], linewidth=2,
        label=f'First-order ($k$={k_equiv:.4f} s$^{{-1}}$)')

# Mark 90% conversion
idx_90_lh = np.argmin(np.abs(X_cstr_lh - 0.9))
idx_90_first = np.argmin(np.abs(X_cstr_first - 0.9))
ax.axhline(y=0.9, color='gray', linestyle=':', alpha=0.5)
ax.plot(tau_range[idx_90_lh], 0.9, 's', color=COLORS['blue'], markersize=12,
        markeredgecolor='black', markeredgewidth=1.5)
ax.plot(tau_range[idx_90_first], 0.9, 's', color=COLORS['vermillion'], markersize=12,
        markeredgecolor='black', markeredgewidth=1.5)

ax.set_xlabel('Space Time, $\\tau$ (s)')
ax.set_ylabel('Conversion, $X$')
ax.set_title('CSTR: LH vs First-Order Kinetics')
ax.set_xlim(0, 300)
ax.set_ylim(0, 1.05)
ax.legend(loc='lower right')

plt.tight_layout()
plt.show()

print(f"Space time for 90% conversion:")
print(f"  LH kinetics:   tau = {tau_range[idx_90_lh]:.0f} s")
print(f"  First-order:   tau = {tau_range[idx_90_first]:.0f} s")

---
## Part 3: PFR Design

The PFR (Plug Flow Reactor) has no axial mixing. Concentration decreases along the reactor length. The design equation is an ODE:

$$\frac{dX}{d\tau} = \frac{r(C_A)}{C_{A0}}, \quad \text{where } C_A = C_{A0}(1-X)$$

**First-order analytical solution:**
$$X = 1 - \exp(-k\tau)$$

In [ ]:
def pfr_conversion_analytical(tau, k, C_A0, order=1):
    """
    Analytical PFR conversion for power-law kinetics.

    Parameters
    ----------
    tau : float or array-like
        Space time (s)
    k : float
        Rate constant (units depend on order)
    C_A0 : float
        Inlet concentration (mol/L)
    order : int
        Reaction order (1 or 2)

    Returns
    -------
    ndarray
        Conversion X (dimensionless)

    Notes
    -----
    First-order:  X = 1 - exp(-k*tau)
    Second-order: X = k*C_A0*tau / (1 + k*C_A0*tau)
    """
    tau = np.asarray(tau, dtype=float)

    if order == 1:
        return 1 - np.exp(-k * tau)
    elif order == 2:
        Da = k * C_A0 * tau
        return Da / (1 + Da)
    else:
        raise ValueError(f"Order {order} not implemented.")


def pfr_ode(tau, X, rate_func, C_A0, rate_args):
    """
    ODE for PFR design equation.

    Parameters
    ----------
    tau : float
        Space time (s)
    X : float
        Conversion
    rate_func : callable
        Rate function r(C_A, *rate_args)
    C_A0 : float
        Inlet concentration (mol/L)
    rate_args : tuple
        Additional arguments for rate_func

    Returns
    -------
    float
        dX/dtau
    """
    C_A = C_A0 * (1 - X[0])
    r = rate_func(C_A, *rate_args)
    return [r / C_A0]


# Define LH rate as function of C_A
def lh_rate_CA(C_A, k_s, K):
    """LH rate as function of concentration."""
    return k_s * K * C_A / (1 + K * C_A)


# Solve PFR with LH kinetics
k_s = 0.05
K_ads = 5.0
C_A0 = 1.0

tau_span = (0, 300)
tau_eval = np.linspace(0, 300, 500)

sol_pfr_lh = solve_ivp(
    pfr_ode,
    tau_span,
    [0.0],  # X(0) = 0
    args=(lh_rate_CA, C_A0, (k_s, K_ads)),
    t_eval=tau_eval,
    method='RK45',
    rtol=1e-10,
    atol=1e-12
)

X_pfr_lh = sol_pfr_lh.y[0]

# Compare with first-order PFR (same equivalent k)
r0 = k_s * K_ads * C_A0 / (1 + K_ads * C_A0)
k_equiv = r0 / C_A0
X_pfr_first = pfr_conversion_analytical(tau_eval, k_equiv, C_A0, order=1)

# Plot
fig, ax = plt.subplots(figsize=(10, 7))

ax.plot(tau_eval, X_pfr_lh, '-', color=COLORS['green'], linewidth=2.5,
        label='PFR with LH kinetics')
ax.plot(tau_eval, X_pfr_first, '--', color=COLORS['vermillion'], linewidth=2,
        label=f'PFR with first-order ($k$={k_equiv:.4f})')

ax.axhline(y=0.9, color='gray', linestyle=':', alpha=0.5)

ax.set_xlabel('Space Time, $\\tau$ (s)')
ax.set_ylabel('Conversion, $X$')
ax.set_title('PFR Performance: LH vs First-Order Kinetics')
ax.set_xlim(0, 300)
ax.set_ylim(0, 1.05)
ax.legend(loc='lower right')

plt.tight_layout()
plt.show()

---
### 3.1 Differential vs Integral Methods for Kinetic Analysis

Two complementary approaches exist for extracting kinetic parameters from batch data:

| Method | Procedure | Best when... |
|--------|-----------|-------------|
| **Integral** | Assume order → integrate → linearize → check fit | Order is suspected (0, 1, 2) |
| **Differential** | Compute $-dC_A/dt$ numerically → log-log plot | Order is unknown or non-integer |

The **integral method** tests candidate orders by checking which linearization gives the best fit:
- Zeroth: $C_A$ vs $t$ (linear)
- First: $\ln C_A$ vs $t$ (linear)
- Second: $1/C_A$ vs $t$ (linear)

The **differential method** uses the power-law form $r = kC_A^n$:
$$\ln\left(-\frac{dC_A}{dt}\right) = \ln k + n\ln C_A$$

A log-log plot of rate vs concentration gives slope = $n$ and intercept = $\ln k$.

In [ ]:
# Differential vs Integral method comparison
# Generate noisy first-order data
rng = np.random.default_rng(42)
k_true = 0.03  # s^-1
C_A0 = 2.0     # mol/L
t_data = np.array([0, 10, 20, 30, 40, 50, 60, 80, 100, 120])
C_exact = C_A0 * np.exp(-k_true * t_data)
C_data = C_exact * (1 + 0.02 * rng.standard_normal(len(t_data)))
C_data[0] = C_A0  # Exact initial condition

# Regressions
res0 = stats.linregress(t_data, C_data)                # Zeroth order
res1 = stats.linregress(t_data, np.log(C_data))        # First order
res2 = stats.linregress(t_data, 1.0 / C_data)          # Second order

# Differential method
dCdt = np.gradient(C_data, t_data)
rate = -dCdt
mask = (rate > 0) & (C_data > 0.05)
mask[0] = False
ln_C = np.log(C_data[mask])
ln_r = np.log(rate[mask])
res_diff = stats.linregress(ln_C, ln_r)

fig, axes = plt.subplot_mosaic(
    [['bar', 'bar'],
     ['integral', 'differential']],
    figsize=(14, 10)
)

# --- Top row: R² bar chart (spans both columns) ---
ax = axes['bar']
bar_labels = ['Zeroth', 'First', 'Second']
r2_vals = [res0.rvalue**2, res1.rvalue**2, res2.rvalue**2]
colors_list = [COLORS['vermillion'], COLORS['blue'], COLORS['green']]
bars = ax.barh(bar_labels, r2_vals, color=colors_list, edgecolor='black', height=0.5)
for bar, r2 in zip(bars, r2_vals):
    ax.text(r2 + 0.01, bar.get_y() + bar.get_height() / 2,
            f'{r2:.4f}', va='center', ha='left', fontsize=11, fontweight='bold')
ax.set_xlim(0, 1.1)
ax.set_xlabel('$R^2$')
ax.set_title('Integral Method: Which Linearization Fits Best?')
ax.axvline(x=1.0, color='gray', linestyle='--', alpha=0.4)

# --- Bottom-left: Best integral fit (first order) ---
ax = axes['integral']
ax.plot(t_data, np.log(C_data), 'o', color=COLORS['blue'], markersize=8,
        markeredgecolor='black', label='Data')
t_fit = np.linspace(0, 120, 100)
ax.plot(t_fit, res1.intercept + res1.slope * t_fit, '-', color=COLORS['blue'],
        linewidth=2, label=f'Fit: $k$ = {-res1.slope:.4f} s$^{{-1}}$')
ax.set_xlabel('Time (s)')
ax.set_ylabel(r'$\ln(C_A)$')
ax.set_title(f'First-Order Linearization ($R^2$ = {res1.rvalue**2:.4f})')
ax.legend()

# --- Bottom-right: Differential method ---
ax = axes['differential']
ax.plot(ln_C, ln_r, 'o', color=COLORS['orange'], markersize=8,
        markeredgecolor='black', label='Numerical $-dC_A/dt$')
x_fit = np.linspace(ln_C.min(), ln_C.max(), 50)
ax.plot(x_fit, res_diff.intercept + res_diff.slope * x_fit, '-',
        color=COLORS['orange'], linewidth=2,
        label=f'Fit: $n$ = {res_diff.slope:.2f}')
ax.set_xlabel(r'$\ln(C_A)$')
ax.set_ylabel(r'$\ln(-dC_A/dt)$')
ax.set_title(f'Differential Method (slope = {res_diff.slope:.2f})')
ax.legend()

plt.tight_layout()
plt.show()

print(f"Integral method: Best fit is FIRST ORDER (R² = {res1.rvalue**2:.4f})")
print(f"  k = {-res1.slope:.4f} s⁻¹ (true: {k_true})")
print(f"Differential method: n = {res_diff.slope:.2f} (true: 1.00)")
print(f"  k = exp({res_diff.intercept:.3f}) = {np.exp(res_diff.intercept):.4f} s⁻¹")
print(f"\nNote: Differential method is noisier due to numerical differentiation,")
print(f"but does not require assuming a reaction order a priori.")

---
## Part 4: CSTR vs PFR Comparison

For positive-order kinetics, the PFR always achieves higher conversion than the CSTR at the same space time. This is because the PFR maintains higher concentrations along most of the reactor length, leading to higher average reaction rates.

The performance gap becomes dramatic at high conversions:
- For 90% conversion with first-order kinetics:
  - PFR: $k\tau = \ln(10) = 2.30$
  - CSTR: $k\tau = 9$  (nearly 4x larger!)

In [ ]:
# CSTR vs PFR comparison for first-order and second-order kinetics
k = 0.05   # s^-1 (first-order) or L/(mol*s) (second-order)
C_A0 = 1.0  # mol/L
tau_range = np.linspace(0, 200, 500)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# --- First-order kinetics ---
X_cstr_1 = cstr_conversion_analytical(tau_range, k, C_A0, order=1)
X_pfr_1 = pfr_conversion_analytical(tau_range, k, C_A0, order=1)

ax1.plot(tau_range, X_pfr_1, '-', color=COLORS['green'], linewidth=2.5,
         label='PFR')
ax1.plot(tau_range, X_cstr_1, '--', color=COLORS['blue'], linewidth=2.5,
         label='CSTR')

# Shade the efficiency gap at tau = 40 s
tau_demo = 40
X_pfr_demo = pfr_conversion_analytical(tau_demo, k, C_A0, order=1)
X_cstr_demo = cstr_conversion_analytical(tau_demo, k, C_A0, order=1)
ax1.annotate('',
             xy=(tau_demo, X_cstr_demo), xytext=(tau_demo, X_pfr_demo),
             arrowprops=dict(arrowstyle='<->', color=COLORS['vermillion'], lw=2))
ax1.text(tau_demo + 3, (X_cstr_demo + X_pfr_demo) / 2,
         f'Gap = {X_pfr_demo - X_cstr_demo:.2f}',
         fontsize=10, color=COLORS['vermillion'])

ax1.axhline(y=0.9, color='gray', linestyle=':', alpha=0.5)
ax1.set_xlabel('Space Time, $\\tau$ (s)')
ax1.set_ylabel('Conversion, $X$')
ax1.set_title('First-Order Kinetics')
ax1.set_ylim(0, 1.05)
ax1.legend(loc='lower right')

# --- Second-order kinetics ---
X_cstr_2 = cstr_conversion_analytical(tau_range, k, C_A0, order=2)
X_pfr_2 = pfr_conversion_analytical(tau_range, k, C_A0, order=2)

ax2.plot(tau_range, X_pfr_2, '-', color=COLORS['green'], linewidth=2.5,
         label='PFR')
ax2.plot(tau_range, X_cstr_2, '--', color=COLORS['blue'], linewidth=2.5,
         label='CSTR')

# Performance gap for second-order
X_pfr_demo2 = pfr_conversion_analytical(tau_demo, k, C_A0, order=2)
X_cstr_demo2 = cstr_conversion_analytical(tau_demo, k, C_A0, order=2)
ax2.annotate('',
             xy=(tau_demo, X_cstr_demo2), xytext=(tau_demo, X_pfr_demo2),
             arrowprops=dict(arrowstyle='<->', color=COLORS['vermillion'], lw=2))
ax2.text(tau_demo + 3, (X_cstr_demo2 + X_pfr_demo2) / 2,
         f'Gap = {X_pfr_demo2 - X_cstr_demo2:.2f}',
         fontsize=10, color=COLORS['vermillion'])

ax2.axhline(y=0.9, color='gray', linestyle=':', alpha=0.5)
ax2.set_xlabel('Space Time, $\\tau$ (s)')
ax2.set_ylabel('Conversion, $X$')
ax2.set_title('Second-Order Kinetics')
ax2.set_ylim(0, 1.05)
ax2.legend(loc='lower right')

plt.tight_layout()
plt.show()

# Quantitative comparison table
print("Space Time Required for Target Conversion")
print("=" * 60)
print(f"{'X':>6} | {'CSTR (1st)':>12} | {'PFR (1st)':>12} | {'Ratio':>8}")
print("-" * 60)
for X_target in [0.50, 0.80, 0.90, 0.95, 0.99]:
    tau_cstr = X_target / (k * (1 - X_target))
    tau_pfr = -np.log(1 - X_target) / k
    ratio = tau_cstr / tau_pfr
    print(f"{X_target:>6.2f} | {tau_cstr:>10.1f} s | {tau_pfr:>10.1f} s | {ratio:>7.2f}x")
print("\nThe CSTR volume penalty grows rapidly at high conversion.")

### ✏️ Concept Check: CSTR vs PFR

**Predict before running the next cell:**

For a **zeroth-order** reaction ($r = k$, independent of $C_A$):
1. Will the PFR still outperform the CSTR?
2. What will the $X$ vs $\tau$ curves look like?

<details>
<summary>Click to reveal answer</summary>

For zeroth-order kinetics, $X = k\tau/C_{A0}$ for **both** CSTR and PFR (until $X = 1$). There is no difference! The PFR advantage arises only because higher concentration → higher rate, but zeroth-order rate is concentration-independent. This is why the CSTR/PFR gap widens with increasing reaction order.
</details>

### 4.1 CSTRs in Series: Approaching PFR Behavior

Multiple CSTRs in series approach PFR performance as the number of stages increases. For $N$ equal-volume CSTRs in series with first-order kinetics:

$$X_N = 1 - \left(\frac{1}{1 + k\tau_i}\right)^N$$

where $\tau_i = \tau_{\text{total}} / N$ is the space time per stage.

In [ ]:
def cstr_series_conversion(tau_total, k, N):
    """
    Conversion for N equal CSTRs in series (first-order kinetics).

    Parameters
    ----------
    tau_total : float or array-like
        Total space time across all reactors (s)
    k : float
        First-order rate constant (s^-1)
    N : int
        Number of CSTRs in series

    Returns
    -------
    ndarray
        Overall conversion X

    Notes
    -----
    Each CSTR has space time tau_i = tau_total / N.
    X_N = 1 - (1 / (1 + k*tau_i))^N
    As N -> infinity, this approaches the PFR solution.
    """
    tau_total = np.asarray(tau_total, dtype=float)
    tau_i = tau_total / N
    return 1 - (1 / (1 + k * tau_i))**N


k = 0.05  # s^-1
tau_range = np.linspace(0, 200, 500)

fig, ax = plt.subplots(figsize=(10, 7))

# PFR (theoretical limit)
X_pfr = pfr_conversion_analytical(tau_range, k, C_A0, order=1)
ax.plot(tau_range, X_pfr, '-', color='black', linewidth=2,
        label='PFR (limit)', alpha=0.8)

# CSTRs in series
N_values = [1, 2, 3, 5, 10]
colors_series = [COLORS['blue'], COLORS['orange'], COLORS['green'],
                 COLORS['vermillion'], COLORS['purple']]

for N, color in zip(N_values, colors_series):
    X_series = cstr_series_conversion(tau_range, k, N)
    ax.plot(tau_range, X_series, '--', color=color, linewidth=2,
            label=f'{N} CSTR{"s" if N > 1 else ""} in series')

ax.axhline(y=0.9, color='gray', linestyle=':', alpha=0.5)
ax.set_xlabel('Total Space Time, $\\tau_{\\mathrm{total}}$ (s)')
ax.set_ylabel('Conversion, $X$')
ax.set_title('CSTRs in Series Approaching PFR Performance')
ax.set_xlim(0, 200)
ax.set_ylim(0, 1.05)
ax.legend(loc='lower right')

plt.tight_layout()
plt.show()

# Print space time for 90% conversion
print("Space time for 90% conversion (first-order, k = 0.05 s^-1):")
print("-" * 50)
tau_pfr_90 = -np.log(0.1) / k
print(f"  PFR:           tau = {tau_pfr_90:.1f} s")
for N in N_values:
    tau_N = N * (1 / (0.1**(1/N)) - 1) / k
    ratio = tau_N / tau_pfr_90
    label = f"{N} CSTR{'s' if N > 1 else ''}"
    print(f"  {label:<15s} tau = {tau_N:.1f} s  ({ratio:.2f}x PFR)")

---
## Part 5: Turnover Frequency (TOF) and Turnover Number (TON)

TOF and TON are site-normalized metrics that allow fair comparison of different catalysts.

**Turnover Frequency (TOF):**
$$\text{TOF} = \frac{\text{moles of product formed}}{\text{moles of active sites} \times \text{time}} \quad [\text{s}^{-1}]$$

**Turnover Number (TON):**
$$\text{TON} = \text{TOF} \times t = \frac{\text{total moles of product}}{\text{moles of active sites}} \quad [\text{dimensionless}]$$

**Active sites** are counted via chemisorption:
$$N_{\text{sites}} = \frac{m_{\text{cat}} \times w_{\text{metal}} \times D}{M_{\text{metal}}}$$

where $D$ is the metal dispersion (fraction of metal atoms on the surface).

In [ ]:
def calculate_tof(rate_mol_per_s, m_cat, w_metal, dispersion, M_metal):
    """
    Calculate turnover frequency from reaction rate and catalyst properties.

    Parameters
    ----------
    rate_mol_per_s : float
        Reaction rate in mol/s (moles of product formed per second)
    m_cat : float
        Mass of catalyst (g)
    w_metal : float
        Metal weight fraction (dimensionless, e.g., 0.02 for 2%)
    dispersion : float
        Metal dispersion (fraction of metal atoms on surface)
    M_metal : float
        Atomic mass of metal (g/mol)

    Returns
    -------
    float
        TOF in s^-1

    Notes
    -----
    N_sites = m_cat * w_metal * dispersion / M_metal
    TOF = rate_mol_per_s / N_sites
    """
    N_sites = m_cat * w_metal * dispersion / M_metal
    tof = rate_mol_per_s / N_sites
    return tof


# Example: Acetylene hydrogenation over Pd/Al2O3
r0 = 4.2e-4        # mol/(L*s), initial rate
V_reactor = 0.500   # L, reactor volume
m_cat = 0.250       # g, catalyst mass
w_Pd = 0.020        # 2% Pd by mass
D_Pd = 0.35         # 35% dispersion
M_Pd = 106.42       # g/mol

# Step 1: Moles converted per unit time
rate_mol = r0 * V_reactor  # mol/s
print("Worked Example: Pd/Al2O3 for Acetylene Hydrogenation")
print("=" * 55)
print(f"Step 1: rate = r0 * V = {r0:.2e} * {V_reactor} = {rate_mol:.2e} mol/s")

# Step 2: Moles of surface Pd
m_Pd = m_cat * w_Pd
n_Pd_total = m_Pd / M_Pd
N_sites = n_Pd_total * D_Pd
print(f"Step 2: m_Pd = {m_Pd:.4f} g")
print(f"        n_Pd = {n_Pd_total:.2e} mol")
print(f"        N_sites = {N_sites:.2e} mol")

# Step 3: TOF
tof = calculate_tof(rate_mol, m_cat, w_Pd, D_Pd, M_Pd)
print(f"Step 3: TOF = {tof:.1f} s^-1 ({tof*60:.0f} min^-1)")
print(f"\nEach surface Pd atom completes ~{tof:.0f} catalytic cycles per second.")

In [ ]:
# TOF vs TON: Effect of catalyst loading
# Simulate a batch reaction for 1 hour with different catalyst loadings
t_reaction = 3600  # seconds
C_A0 = 0.1  # mol/L
V = 1.0  # L

# Assume first-order kinetics with rate proportional to catalyst mass
# r = k' * m_cat * C_A where k' = 0.001 L/(g*s)
k_prime = 0.001  # L/(g*s)
w_metal = 0.05   # 5% loading
D_metal = 0.30   # 30% dispersion
M_metal = 195.08  # Pt atomic mass

m_cat_range = np.linspace(0.1, 5.0, 20)  # grams of catalyst

tof_values = []
ton_values = []
X_values = []

for m in m_cat_range:
    k_eff = k_prime * m  # Effective first-order rate constant
    X = 1 - np.exp(-k_eff * t_reaction)  # PFR-like batch conversion
    X_values.append(X)

    moles_product = C_A0 * V * X
    rate_avg = moles_product / t_reaction
    N_sites = m * w_metal * D_metal / M_metal

    tof = rate_avg / N_sites
    ton = moles_product / N_sites
    tof_values.append(tof)
    ton_values.append(ton)

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(16, 7))

# TOF vs catalyst loading
ax1.plot(m_cat_range, tof_values, 'o-', color=COLORS['blue'], markersize=6)
ax1.set_xlabel('Catalyst Mass (g)')
ax1.set_ylabel('TOF (s$^{-1}$)')
ax1.set_title('TOF vs Catalyst Loading')

# TON vs catalyst loading
ax2.plot(m_cat_range, ton_values, 's-', color=COLORS['green'], markersize=6)
ax2.set_xlabel('Catalyst Mass (g)')
ax2.set_ylabel('TON (dimensionless)')
ax2.set_title('TON vs Catalyst Loading')

# Conversion vs catalyst loading
ax3.plot(m_cat_range, X_values, '^-', color=COLORS['orange'], markersize=6)
ax3.set_xlabel('Catalyst Mass (g)')
ax3.set_ylabel('Conversion, $X$')
ax3.set_title('Conversion vs Catalyst Loading')
ax3.set_ylim(0, 1.05)

plt.tight_layout()
plt.show()

print("Key Observation:")
print("- TOF here is the AVERAGE over 1 hour, not the initial TOF")
print("- Average TOF decreases with more catalyst because conversion")
print("  approaches 100%, making the average rate lower per site")
print("- The INITIAL (intrinsic) TOF = k' × C_A0 is constant regardless of loading")
print("- TON also decreases at high loading due to diminishing returns")
print("\nImportant: Standard practice uses initial rates for TOF.")
print("Average TOF conflates catalyst activity with reactor conversion.")

---
## Part 6: Space Time and Space Velocity

In industrial practice, reactor throughput is expressed as **space velocity** rather than space time.

| Metric | Definition | Units |
|--------|-----------|-------|
| **WHSV** | mass flow rate of feed / mass of catalyst | h$^{-1}$ |
| **GHSV** | volumetric gas flow rate (STP) / bed volume | h$^{-1}$ |
| **LHSV** | liquid volumetric flow rate / bed volume | h$^{-1}$ |

Space velocity is the reciprocal of contact time (with appropriate conversions). Higher space velocity means shorter contact time.

In [ ]:
def whsv_to_tau(whsv, rho_feed=1.0):
    """
    Convert WHSV to space time (approximate).

    Parameters
    ----------
    whsv : float
        Weight hourly space velocity (h^-1)
    rho_feed : float
        Feed density (g/L or kg/m^3). Default 1.0.

    Returns
    -------
    float
        Space time in seconds
    """
    return 3600 / whsv  # Approximate: 1/WHSV in hours -> seconds


def ghsv_to_tau(ghsv):
    """
    Convert GHSV to contact time.

    Parameters
    ----------
    ghsv : float
        Gas hourly space velocity (h^-1)

    Returns
    -------
    float
        Contact time in seconds
    """
    return 3600 / ghsv


# Example calculations for common industrial processes
print("Space Velocity Examples for Industrial Catalytic Processes")
print("=" * 65)
print(f"{'Process':<30} {'WHSV (h^-1)':>12} {'tau (s)':>10}")
print("-" * 65)

processes = [
    ('Hydrodesulfurization (HDS)', 2.0),
    ('FCC Cracking', 15.0),
    ('Methanol Synthesis', 4.0),
    ('Fischer-Tropsch', 1.0),
    ('Ammonia Synthesis', 5.0),
]

for name, whsv in processes:
    tau = whsv_to_tau(whsv)
    print(f"{name:<30} {whsv:>12.1f} {tau:>10.0f}")

print("\n" + "=" * 65)
print("\nAutomotive Catalytic Converter (monolith PFR):")
ghsv_auto = 50000  # h^-1, typical at highway speed
tau_auto = ghsv_to_tau(ghsv_auto)
print(f"  GHSV = {ghsv_auto:,} h^-1")
print(f"  Contact time = {tau_auto*1000:.1f} ms")
print(f"  Challenge: >95% conversion of CO, HC, NOx in {tau_auto*1000:.0f} ms!")

---
## Part 7: Interactive Parameter Exploration

Let us explore how the key kinetic and design parameters affect reactor performance.

In [ ]:
# =============================================================================
# ADJUSTABLE PARAMETERS - Students can modify these values!
# =============================================================================

# Kinetic parameters
k_value = 0.05           # Rate constant (s^-1 for first order)
reaction_order = 1       # Try 1 or 2
C_A0 = 1.0               # Inlet concentration (mol/L)
tau_max = 200             # Maximum space time to plot (s)

# LH parameters (for LH comparison)
k_s_value = 0.08         # Surface rate constant (mol/(L*s))
K_ads_value = 3.0        # Adsorption constant (L/mol)

# =============================================================================
# COMPUTATION AND PLOTTING
# =============================================================================

tau_range = np.linspace(0.1, tau_max, 500)

# Power-law kinetics
X_cstr_pl = cstr_conversion_analytical(tau_range, k_value, C_A0, order=reaction_order)
X_pfr_pl = pfr_conversion_analytical(tau_range, k_value, C_A0, order=reaction_order)

# LH kinetics (PFR via ODE)
sol_pfr = solve_ivp(
    pfr_ode, (0, tau_max), [0.0],
    args=(lh_rate_CA, C_A0, (k_s_value, K_ads_value)),
    t_eval=tau_range, method='RK45', rtol=1e-10, atol=1e-12
)
X_pfr_lh_int = sol_pfr.y[0]

# LH kinetics (CSTR via fsolve)
X_cstr_lh_int = np.zeros_like(tau_range)
for i, tau_val in enumerate(tau_range):
    X_sol = fsolve(cstr_lh_residual, 0.5,
                   args=(tau_val, k_s_value, K_ads_value, C_A0))
    X_cstr_lh_int[i] = np.clip(X_sol[0], 0, 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Power-law comparison
ax1.plot(tau_range, X_pfr_pl, '-', color=COLORS['green'], linewidth=2.5,
         label='PFR')
ax1.plot(tau_range, X_cstr_pl, '--', color=COLORS['blue'], linewidth=2.5,
         label='CSTR')
ax1.set_xlabel('Space Time, $\\tau$ (s)')
ax1.set_ylabel('Conversion, $X$')
ax1.set_title(f'Power-Law (order={reaction_order}, $k$={k_value})')
ax1.set_ylim(0, 1.05)
ax1.legend(loc='lower right')

# LH comparison
ax2.plot(tau_range, X_pfr_lh_int, '-', color=COLORS['green'], linewidth=2.5,
         label='PFR')
ax2.plot(tau_range, X_cstr_lh_int, '--', color=COLORS['blue'], linewidth=2.5,
         label='CSTR')
ax2.set_xlabel('Space Time, $\\tau$ (s)')
ax2.set_ylabel('Conversion, $X$')
ax2.set_title(f'LH Kinetics ($k_s$={k_s_value}, $K$={K_ads_value})')
ax2.set_ylim(0, 1.05)
ax2.legend(loc='lower right')

plt.tight_layout()
plt.show()

print("Try modifying the parameters above:")
print("  - Change reaction_order to 2 and observe the larger CSTR/PFR gap")
print("  - Increase K_ads_value to see stronger surface saturation effects")
print("  - Double k_value and see how tau requirements change")

---
### 7.1 Temperature Effects on Reactor Performance

How does reactor conversion change with temperature? Using the Arrhenius equation $k(T) = A\exp(-E_a/RT)$, we can predict performance across a temperature range.

This connects **Chapter 2** (temperature dependence) with **Chapter 3** (reactor design).

In [ ]:
# ADJUSTABLE PARAMETERS - Students can modify these values!
# =============================================================================
E_a = 80e3        # Activation energy (J/mol)
A = 1e8           # Pre-exponential factor (s^-1, first order)
C_A0 = 1.0        # Inlet concentration (mol/L)
tau_design = 100  # Design space time (s)
T_range = np.linspace(300, 600, 200)  # Temperature range (K)
# =============================================================================

R_gas = 8.314
k_T = A * np.exp(-E_a / (R_gas * T_range))

# CSTR and PFR conversion vs temperature
X_cstr = k_T * tau_design / (1 + k_T * tau_design)
X_pfr = 1 - np.exp(-k_T * tau_design)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Panel 1: k(T) on log scale
ax1.semilogy(T_range, k_T, '-', color=COLORS['blue'], linewidth=2.5)
ax1.set_xlabel('Temperature (K)')
ax1.set_ylabel('$k$ (s$^{-1}$)')
ax1.set_title(f'Arrhenius: $E_a$ = {E_a/1000:.0f} kJ/mol')
ax1.axhline(y=1/tau_design, color=COLORS['vermillion'], linestyle='--',
            label=f'$k = 1/\\tau$ = {1/tau_design:.3f} s$^{{-1}}$')
ax1.legend()

# Panel 2: Conversion vs temperature
ax2.plot(T_range, X_pfr, '-', color=COLORS['green'], linewidth=2.5, label='PFR')
ax2.plot(T_range, X_cstr, '--', color=COLORS['blue'], linewidth=2.5, label='CSTR')
ax2.set_xlabel('Temperature (K)')
ax2.set_ylabel('Conversion, $X$')
ax2.set_title(f'Reactor Performance ($\\tau$ = {tau_design} s)')
ax2.set_ylim(0, 1.05)
ax2.axhline(y=0.90, color='gray', linestyle=':', alpha=0.5, label='$X$ = 0.90')
# Find T for 90% conversion
T_90_pfr = T_range[np.argmin(np.abs(X_pfr - 0.90))]
T_90_cstr = T_range[np.argmin(np.abs(X_cstr - 0.90))]
ax2.axvline(x=T_90_pfr, color=COLORS['green'], linestyle=':', alpha=0.5)
ax2.axvline(x=T_90_cstr, color=COLORS['blue'], linestyle=':', alpha=0.5)
ax2.legend()

plt.tight_layout()
plt.show()

print(f"Temperature for 90% conversion:")
print(f"  PFR:  T ≈ {T_90_pfr:.0f} K")
print(f"  CSTR: T ≈ {T_90_cstr:.0f} K  (needs {T_90_cstr - T_90_pfr:.0f} K higher)")
print(f"\nTry changing E_a and tau_design to see how they affect these temperatures.")

### ✏️ Concept Check: Temperature and Reactor Design

**Before modifying the parameters above:**

1. If you double $E_a$ (from 80 to 160 kJ/mol), will the temperature gap between PFR and CSTR (for 90% conversion) increase or decrease?

2. If you increase $\tau$ by 10×, roughly how much can you lower the operating temperature while maintaining 90% conversion?

<details>
<summary>Click to reveal answers</summary>

1. **Increase.** Higher $E_a$ means $k(T)$ is more sensitive to temperature. At any given $T$, the CSTR/PFR conversion gap is the same fraction of $X$, but since the $X$ vs $T$ curve becomes steeper, the temperature difference needed to compensate becomes larger.

2. Increasing $\tau$ by 10× means $k\tau$ stays the same when $k$ is 10× smaller. From Arrhenius: $\Delta T \approx RT^2 \ln(10)/E_a$. At $T \approx 450$ K with $E_a = 80$ kJ/mol: $\Delta T \approx 8.314 \times 450^2 \times 2.303 / 80000 \approx 49$ K lower. Try it!
</details>

### Key Takeaways — Part I: Ideal Reactor Design

- CSTR: algebraic design equation; conversion limited by exit conditions
- PFR: ODE-based design; always achieves higher conversion than CSTR for positive-order kinetics
- N CSTRs in series approach PFR behavior as N increases
- TOF normalizes activity per active site; TON measures total catalytic cycles

> **Session Break (suggested):** Good stopping point between reactor design and characterization. Save your work before continuing.

---
## Part 8: Catalyst Characterization -- Introduction

Reactor design equations tell us *how fast* reactions proceed -- but how do we know the catalyst's surface area, binding energies, and crystallite structure? Part II introduces the key experimental techniques.

Catalyst characterization answers three fundamental questions:
1. **How much surface area is available?** (BET, chemisorption)
2. **How strongly do species bind?** (TPD, calorimetry)
3. **What is the catalyst structure?** (XRD, spectroscopy)

### 8.1 BET Surface Area Analysis

The Brunauer-Emmett-Teller (BET) method determines total surface area from N$_2$ physisorption data at 77 K. The method works by:

1. Measuring the volume of N$_2$ adsorbed at various relative pressures $P/P_0$
2. Linearizing the data using the BET transform
3. Extracting the monolayer volume $V_m$ and BET constant $c$ from the slope and intercept
4. Computing surface area from $V_m$

The BET equation in linear form:
$$\frac{P/P_0}{V(1 - P/P_0)} = \frac{1}{V_m c} + \frac{c - 1}{V_m c} \cdot \frac{P}{P_0}$$

This is $y = b + m \cdot x$ where:
- $y = (P/P_0) / [V(1 - P/P_0)]$
- $x = P/P_0$
- Slope $m = (c-1)/(V_m c)$
- Intercept $b = 1/(V_m c)$

In [ ]:
# Generate synthetic BET data for a mesoporous silica catalyst
# (mimicking a Type II isotherm with S_BET ~ 250 m^2/g)

# Known parameters for data generation
V_m_true = 57.5        # cm^3(STP)/g, monolayer volume
c_true = 120.0         # BET constant
m_sample = 0.150       # g, sample mass

# Relative pressure points
p_over_p0 = np.array([
    0.02, 0.04, 0.06, 0.08, 0.10,
    0.12, 0.15, 0.18, 0.20, 0.22,
    0.25, 0.28, 0.30, 0.33, 0.35,
    0.40, 0.45, 0.50, 0.55, 0.60,
    0.65, 0.70, 0.75, 0.80, 0.85,
    0.90, 0.95
])

# BET isotherm: V = V_m * c * (P/P0) / ((1 - P/P0) * (1 + (c-1) * P/P0))
V_ads_ideal = V_m_true * c_true * p_over_p0 / (
    (1 - p_over_p0) * (1 + (c_true - 1) * p_over_p0)
)

# Add realistic noise (larger at higher pressures due to capillary condensation)
np.random.seed(42)  # Reproducibility
noise = np.random.normal(0, 0.5, len(p_over_p0)) * (1 + 2 * p_over_p0)
V_ads = V_ads_ideal + noise

# Create DataFrame
bet_data = pd.DataFrame({
    'P_over_P0': p_over_p0,
    'V_ads_cm3_g': V_ads
})

print("N2 Adsorption Data (Mesoporous Silica, 77 K)")
print("=" * 45)
print(f"Sample mass: {m_sample} g")
print(f"Number of data points: {len(bet_data)}")
print()
print(bet_data.to_string(index=False, float_format='{:.3f}'.format))

In [ ]:
# Plot the raw adsorption isotherm
fig, ax = plt.subplots(figsize=(10, 7))

ax.plot(p_over_p0, V_ads, 'o-', color=COLORS['blue'], markersize=8,
        markeredgecolor='black', markeredgewidth=1,
        label='N$_2$ adsorption at 77 K')

# Mark the BET region
bet_mask = (p_over_p0 >= 0.05) & (p_over_p0 <= 0.30)
ax.axvspan(0.05, 0.30, alpha=0.1, color=COLORS['green'],
           label='BET region (0.05-0.30)')

# Mark approximate Point B (knee of isotherm, ~ V_m)
idx_B = np.argmin(np.abs(V_ads - V_m_true))
ax.plot(p_over_p0[idx_B], V_ads[idx_B], 's', color=COLORS['vermillion'],
        markersize=14, markeredgecolor='black', markeredgewidth=1.5,
        label=f'Point B ($\\approx V_m$)', zorder=5)

ax.set_xlabel('Relative Pressure, $P/P_0$')
ax.set_ylabel('Volume Adsorbed, $V$ (cm$^3$(STP)/g)')
ax.set_title('N$_2$ Adsorption Isotherm (Type II)')
ax.set_xlim(0, 1)
ax.set_ylim(0, None)
ax.legend(loc='upper left')

plt.tight_layout()
plt.show()

print("The isotherm shows a Type II shape:")
print("  - Steep initial rise: first layer forming")
print("  - Knee (Point B): approximate monolayer completion")
print("  - Gradual rise: multilayer buildup")
print("  - Steep final rise: condensation in large pores")

### 8.2 BET Transform and Linear Regression

In [ ]:
def bet_transform(p_over_p0, V_ads):
    """
    Apply BET linearization transform to adsorption data.

    Parameters
    ----------
    p_over_p0 : array-like
        Relative pressure P/P0 (dimensionless)
    V_ads : array-like
        Volume adsorbed (cm^3(STP)/g)

    Returns
    -------
    x_bet : ndarray
        P/P0 values (x-axis of BET plot)
    y_bet : ndarray
        (P/P0) / [V * (1 - P/P0)] values (y-axis of BET plot)
    """
    p_over_p0 = np.asarray(p_over_p0)
    V_ads = np.asarray(V_ads)
    x_bet = p_over_p0
    y_bet = p_over_p0 / (V_ads * (1 - p_over_p0))
    return x_bet, y_bet


def bet_analysis(p_over_p0, V_ads, p_min=0.05, p_max=0.30):
    """
    Perform complete BET analysis on adsorption isotherm data.

    Parameters
    ----------
    p_over_p0 : array-like
        Relative pressure P/P0
    V_ads : array-like
        Volume adsorbed (cm^3(STP)/g)
    p_min : float, optional
        Lower bound of BET range (default 0.05)
    p_max : float, optional
        Upper bound of BET range (default 0.30)

    Returns
    -------
    dict
        Results containing V_m, c, S_BET, R_squared, slope, intercept,
        x_bet, y_bet

    Notes
    -----
    Uses sigma_N2 = 0.162 nm^2 = 16.2e-20 m^2 for N2 at 77 K.
    Simplified formula: S_BET (m^2/g) = 4.35 * V_m (cm^3(STP)/g).
    """
    p_over_p0 = np.asarray(p_over_p0)
    V_ads = np.asarray(V_ads)

    # Select data in BET range
    mask = (p_over_p0 >= p_min) & (p_over_p0 <= p_max)
    x_bet, y_bet = bet_transform(p_over_p0[mask], V_ads[mask])

    # Linear regression
    result = stats.linregress(x_bet, y_bet)
    slope = result.slope
    intercept = result.intercept
    r_squared = result.rvalue**2

    # Extract BET parameters
    V_m = 1 / (slope + intercept)
    c = 1 + slope / intercept

    # =========================================================================
    # ADJUSTABLE PARAMETERS — BET surface area constants
    # =========================================================================
    sigma_N2 = 16.2e-20  # m^2, N2 molecular cross-section at 77 K
    # N_A and V_mol_STP are defined in the setup cell above
    # =========================================================================
    S_BET = V_m * N_A * sigma_N2 / V_mol_STP  # m^2/g
    # Equivalent to: S_BET = 4.35 * V_m

    return {
        'V_m': V_m,
        'c': c,
        'S_BET': S_BET,
        'R_squared': r_squared,
        'slope': slope,
        'intercept': intercept,
        'x_bet': x_bet,
        'y_bet': y_bet
    }


# Perform BET analysis
results = bet_analysis(p_over_p0, V_ads)

print("BET Analysis Results")
print("=" * 50)
print(f"Slope     = {results['slope']:.6f} g/cm^3")
print(f"Intercept = {results['intercept']:.6f} g/cm^3")
print(f"R-squared = {results['R_squared']:.6f}")
print()
print(f"Monolayer volume, V_m = {results['V_m']:.2f} cm^3(STP)/g")
print(f"BET constant, c       = {results['c']:.1f}")
print(f"Surface area, S_BET   = {results['S_BET']:.1f} m^2/g")
print(f"  (Simplified: 4.35 * V_m = {4.35 * results['V_m']:.1f} m^2/g)")

# Validity checks
print("\nValidity Checks:")
print(f"  R^2 > 0.999?  {'Yes' if results['R_squared'] > 0.999 else 'No'} ({results['R_squared']:.4f})")
print(f"  Intercept > 0? {'Yes' if results['intercept'] > 0 else 'No'} ({results['intercept']:.6f})")
print(f"  c > 2?         {'Yes' if results['c'] > 2 else 'No'} (c = {results['c']:.1f})")

In [ ]:
# Publication-quality BET plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# --- Left panel: BET linear plot ---
x_all, y_all = bet_transform(p_over_p0, V_ads)
bet_mask = (p_over_p0 >= 0.05) & (p_over_p0 <= 0.30)

ax1.plot(x_all[~bet_mask], y_all[~bet_mask], 'o', color='gray',
         markersize=7, alpha=0.4, label='Outside BET range')
ax1.plot(results['x_bet'], results['y_bet'], 's', color=COLORS['blue'],
         markersize=10, markeredgecolor='black', markeredgewidth=1.5,
         label='BET range (0.05-0.30)')

x_fit = np.linspace(0, 0.35, 100)
y_fit = results['intercept'] + results['slope'] * x_fit
ax1.plot(x_fit, y_fit, '-', color=COLORS['vermillion'], linewidth=2,
         label=f'Linear fit ($R^2$ = {results["R_squared"]:.4f})')

textstr = (f'$V_m$ = {results["V_m"]:.1f} cm$^3$/g\n'
           f'$c$ = {results["c"]:.0f}\n'
           f'$S_{{\\mathrm{{BET}}}}$ = {results["S_BET"]:.0f} m$^2$/g')
props = dict(boxstyle='round', facecolor='wheat', alpha=0.8)
ax1.text(0.05, 0.95, textstr, transform=ax1.transAxes, fontsize=11,
         verticalalignment='top', bbox=props)

ax1.set_xlabel('$P/P_0$')
ax1.set_ylabel('$(P/P_0) / [V(1-P/P_0)]$ (g/cm$^3$)')
ax1.set_title('BET Linearization Plot')
ax1.set_xlim(-0.02, 0.38)
ax1.legend(loc='lower right', fontsize=10)

# --- Right panel: Raw isotherm with V_m marked ---
ax2.plot(p_over_p0, V_ads, 'o-', color=COLORS['blue'], markersize=6,
         markeredgecolor='black', markeredgewidth=0.5,
         label='Adsorption data')
ax2.axhline(y=results['V_m'], color=COLORS['orange'], linestyle='--',
            linewidth=2, label=f'$V_m$ = {results["V_m"]:.1f} cm$^3$/g')
ax2.axvspan(0.05, 0.30, alpha=0.1, color=COLORS['green'])

ax2.set_xlabel('$P/P_0$')
ax2.set_ylabel('$V_{\\mathrm{ads}}$ (cm$^3$(STP)/g)')
ax2.set_title('N$_2$ Adsorption Isotherm')
ax2.set_xlim(0, 1)
ax2.set_ylim(0, None)
ax2.legend(loc='upper left')

plt.tight_layout()
plt.show()

### ✏️ Concept Check: BET Analysis

1. You measure $V_m = 120$ cm³(STP)/g. Estimate $S_{\text{BET}}$ without a calculator. *(Hint: use the 4.35 factor.)*

2. Your BET plot has an intercept of $-0.001$. Is the analysis valid?

3. A zeolite gives $c = 500$ and $S_{\text{BET}} = 800$ m²/g. Should you trust this value?

<details>
<summary>Click to reveal answers</summary>

1. $S_{\text{BET}} \approx 4.35 \times 120 = 522$ m²/g.

2. **No.** A negative intercept gives a negative $c$, which violates the BET physical model. Re-run with a narrower $P/P_0$ window (try 0.05–0.15).

3. **Be cautious.** Zeolites are microporous (Type I isotherm), where BET overestimates surface area because the model assumes multilayer adsorption but micropore filling occurs instead. Use $t$-plot or Langmuir analysis for microporous materials.
</details>

---
## Part 9: TPD Analysis

Temperature-Programmed Desorption (TPD) measures binding energies by heating an adsorbate-covered surface at a constant rate and monitoring the desorbed species.

**First-order desorption rate:**
$$r_{\text{des}} = \nu \cdot \theta \cdot \exp\left(-\frac{E_d}{RT}\right)$$

**Redhead equation** (for estimating $E_d$ from $T_p$):
$$E_d = RT_p\left[\ln\left(\frac{\nu T_p}{\beta}\right) - 3.64\right]$$

**Rule of thumb:** $E_d$ (kJ/mol) $\approx 0.25 \times T_p$ (K) when $\beta = 10$ K/s, or $\approx 0.28 \times T_p$ (K) when $\beta = 10$ K/min. The difference arises because smaller $\beta$ gives a larger $\ln(\nu T_p / \beta)$ term.

In [ ]:
def redhead_equation(T_peak, nu=1e13, beta=10.0):
    """
    Calculate desorption energy using the Redhead equation.

    Parameters
    ----------
    T_peak : float
        Temperature at peak maximum (K)
    nu : float, optional
        Pre-exponential factor (s^-1). Default 1e13 s^-1.
    beta : float, optional
        Heating rate (K/s). Default 10.0 K/s.
        Note: For beta in K/min, divide by 60 first.

    Returns
    -------
    float
        Desorption activation energy (J/mol)

    Notes
    -----
    Valid when E_d/(R*T_p) is between 20 and 50.
    Rule of thumb: E_d (kJ/mol) ~ 0.25 * T_p (K) for beta = 10 K/s,
    or ~ 0.28 * T_p (K) for beta = 10 K/min.
    """
    return R * T_peak * (np.log(nu * T_peak / beta) - 3.64)


def asymmetric_gaussian(T, T_center, sigma, amplitude, asymmetry=0.3):
    """
    Generate an asymmetric Gaussian peak for TPD simulation.

    Parameters
    ----------
    T : array-like
        Temperature array (K)
    T_center : float
        Peak center temperature (K)
    sigma : float
        Peak width parameter (K)
    amplitude : float
        Peak height
    asymmetry : float, optional
        Asymmetry factor (0 = symmetric). Default 0.3.

    Returns
    -------
    ndarray
        Simulated desorption signal
    """
    T = np.asarray(T)
    gaussian = np.exp(-((T - T_center)**2) / (2 * sigma**2))
    asym_factor = 1 + asymmetry * np.tanh((T - T_center) / 20.0)
    return amplitude * gaussian * asym_factor


# Demonstrate the Redhead equation with both beta conventions
print("Redhead Equation: Quick Reference")
print("=" * 65)
print(f"Assumed: nu = 10^13 s^-1\n")
print(f"{'T_p (K)':>10} | {'beta=10 K/s':>14} | {'0.25*T_p':>10} | "
      f"{'beta=10 K/min':>14} | {'0.28*T_p':>10}")
print("-" * 65)

for T_p in [300, 400, 500, 600, 700, 800]:
    E_d_Ks = redhead_equation(T_p, nu=1e13, beta=10.0)        # beta = 10 K/s
    E_d_Kmin = redhead_equation(T_p, nu=1e13, beta=10.0/60)   # beta = 10 K/min
    print(f"{T_p:>10} | {E_d_Ks/1000:>14.1f} | {0.25*T_p:>10.0f} | "
          f"{E_d_Kmin/1000:>14.1f} | {0.28*T_p:>10.0f}")

print("\nThe rule of thumb E_d ~ 0.25*T_p applies for beta = 10 K/s.")
print("The rule of thumb E_d ~ 0.28*T_p applies for beta = 10 K/min.")

In [ ]:
# Generate a synthetic TPD spectrum with two binding states
T_min, T_max = 300, 800
T = np.linspace(T_min, T_max, 1000)

# Weak binding site (alpha state)
T_peak_alpha = 420; sigma_alpha = 28; amplitude_alpha = 0.6
# Strong binding site (beta state)
T_peak_beta = 600; sigma_beta = 35; amplitude_beta = 1.0

# Experimental conditions
nu = 1e13                # Pre-exponential factor (s^-1)
beta_Kmin = 10.0         # Heating rate (K/min)
beta_Ks = beta_Kmin / 60 # K/s

signal_alpha = asymmetric_gaussian(T, T_peak_alpha, sigma_alpha, amplitude_alpha)
signal_beta = asymmetric_gaussian(T, T_peak_beta, sigma_beta, amplitude_beta)
signal_total = signal_alpha + signal_beta

idx_alpha = np.argmax(signal_alpha)
idx_beta = np.argmax(signal_beta)
T_p_alpha = T[idx_alpha]
T_p_beta = T[idx_beta]

E_d_alpha = redhead_equation(T_p_alpha, nu=nu, beta=beta_Ks)
E_d_beta = redhead_equation(T_p_beta, nu=nu, beta=beta_Ks)

# Plot TPD spectrum
fig, ax = plt.subplots(figsize=(11, 7))
ax.plot(T, signal_alpha, '--', color=COLORS['orange'], linewidth=2.5,
        label=f'$\\alpha$ (weak): $E_d$ = {E_d_alpha/1000:.1f} kJ/mol')
ax.plot(T, signal_beta, '-', color=COLORS['blue'], linewidth=2.5,
        label=f'$\\beta$ (strong): $E_d$ = {E_d_beta/1000:.1f} kJ/mol')
ax.plot(T, signal_total, '-', color='gray', linewidth=2, alpha=0.5,
        label='Total spectrum')
ax.axvline(x=T_p_alpha, color=COLORS['orange'], linestyle=':', linewidth=1.5, alpha=0.7)
ax.axvline(x=T_p_beta, color=COLORS['blue'], linestyle=':', linewidth=1.5, alpha=0.7)

ax.annotate(f'$T_p^\\alpha$ = {T_p_alpha:.0f} K',
            xy=(T_p_alpha, signal_alpha[idx_alpha]),
            xytext=(T_p_alpha - 50, signal_alpha[idx_alpha] + 0.12),
            fontsize=11, color=COLORS['orange'],
            arrowprops=dict(arrowstyle='->', color=COLORS['orange']))
ax.annotate(f'$T_p^\\beta$ = {T_p_beta:.0f} K',
            xy=(T_p_beta, signal_beta[idx_beta]),
            xytext=(T_p_beta + 30, signal_beta[idx_beta] + 0.12),
            fontsize=11, color=COLORS['blue'],
            arrowprops=dict(arrowstyle='->', color=COLORS['blue']))

textbox = (f"Redhead Analysis\n"
           f"$\\nu$ = {nu:.0e} s$^{{-1}}$\n"
           f"$\\beta$ = {beta_Kmin:.0f} K/min\n"
           f"$E_d = RT_p[\\ln(\\nu T_p/\\beta) - 3.64]$")
props = dict(boxstyle='round', facecolor='white', edgecolor='gray', alpha=0.9)
ax.text(0.02, 0.98, textbox, transform=ax.transAxes, fontsize=10,
        verticalalignment='top', fontfamily='monospace', bbox=props)

ax.set_xlabel('Temperature (K)')
ax.set_ylabel('Desorption Rate (a.u.)')
ax.set_title('Temperature-Programmed Desorption (TPD) Spectrum')
ax.set_xlim(T_min, T_max)
ax.set_ylim(0, None)
ax.legend(loc='upper right')
plt.tight_layout()
plt.show()

print("TPD Analysis Results")
print("=" * 55)
print(f"Heating rate: {beta_Kmin:.0f} K/min ({beta_Ks:.4f} K/s)")
print(f"Alpha state (weak):  T_p = {T_p_alpha:.0f} K, E_d = {E_d_alpha/1000:.1f} kJ/mol")
print(f"Beta state (strong): T_p = {T_p_beta:.0f} K, E_d = {E_d_beta/1000:.1f} kJ/mol")
print(f"\nRule-of-thumb check (0.28*T_p for beta = 10 K/min):")
print(f"  Alpha: 0.28 * {T_p_alpha:.0f} = {0.28*T_p_alpha:.0f} kJ/mol (actual: {E_d_alpha/1000:.1f})")
print(f"  Beta:  0.28 * {T_p_beta:.0f} = {0.28*T_p_beta:.0f} kJ/mol (actual: {E_d_beta/1000:.1f})")

### ✏️ Concept Check: TPD Interpretation

A CO-TPD experiment on a bimetallic PtRu catalyst shows three peaks at $T_p$ = 350, 480, and 620 K (heating rate 10 K/min).

1. Estimate the desorption energies using the rule of thumb $E_d \approx 0.28 \times T_p$.
2. Which peak likely corresponds to CO on Ru sites? *(Hint: Ru binds CO more strongly than Pt.)*
3. If you increase the heating rate to 20 K/min, will the peak temperatures shift higher or lower?

<details>
<summary>Click to reveal answers</summary>

1. $E_d \approx$ 98, 134, 174 kJ/mol for the three peaks.

2. The 620 K peak (174 kJ/mol) most likely corresponds to CO on Ru — Ru binds CO more strongly due to greater back-donation into CO π* orbitals. The 350 K peak may be weakly bound CO (e.g., on terrace sites), and 480 K is typical of CO on Pt.

3. **Higher.** Faster heating means less time for desorption at each temperature, so the surface must reach a higher temperature before the desorption rate can "catch up." This is why the Redhead equation explicitly includes $\beta$.
</details>

---
## Part 10: XRD Analysis and the Scherrer Equation

X-ray Diffraction (XRD) identifies crystalline phases and determines crystallite size.

**Bragg's Law:** $n\lambda = 2d\sin\theta$

**Scherrer Equation** for crystallite size from peak broadening:
$$D = \frac{K\lambda}{\beta\cos\theta}$$

where $K \approx 0.9$, $\lambda$ is the X-ray wavelength (Cu K$\alpha$ = 0.15406 nm), $\beta$ is the peak FWHM in radians, and $\theta$ is the Bragg angle.

In [ ]:
def scherrer_size(two_theta_deg, fwhm_deg, wavelength_nm=0.15406, K=0.9):
    """
    Calculate crystallite size using the Scherrer equation.

    Parameters
    ----------
    two_theta_deg : float
        Peak position in 2-theta (degrees)
    fwhm_deg : float
        Full width at half maximum (degrees)
    wavelength_nm : float, optional
        X-ray wavelength in nm. Default Cu K-alpha = 0.15406 nm.
    K : float, optional
        Shape factor. Default 0.9.

    Returns
    -------
    float
        Crystallite size D in nm
    """
    theta_rad = np.radians(two_theta_deg / 2)
    beta_rad = np.radians(fwhm_deg)
    D = K * wavelength_nm / (beta_rad * np.cos(theta_rad))
    return D


def generate_xrd_peak(two_theta_array, two_theta_center, intensity, fwhm_deg):
    """Generate a Gaussian XRD peak."""
    sigma = fwhm_deg / (2 * np.sqrt(2 * np.log(2)))
    return intensity * np.exp(-((two_theta_array - two_theta_center)**2)
                              / (2 * sigma**2))


# =========================================================================
# ADJUSTABLE PARAMETERS — Scherrer / XRD constants
# =========================================================================
# Pt fcc reference peaks (Cu K-alpha)
pt_peaks = {
    '(111)': {'2theta': 39.76, 'rel_intensity': 100},
    '(200)': {'2theta': 46.24, 'rel_intensity': 53},
    '(220)': {'2theta': 67.45, 'rel_intensity': 31},
}
D_crystallite = 5.0  # nm — try 2, 5, 10, 20 to see peak broadening
# wavelength_nm and K are set via scherrer_size() defaults above
# =========================================================================

# Generate synthetic XRD pattern for Pt/Al2O3
two_theta = np.linspace(20, 80, 2000)

xrd_signal = np.zeros_like(two_theta)
background = 50 * np.exp(-((two_theta - 45)**2) / (2 * 15**2)) + 20
xrd_signal += background

peak_data = []
for hkl, info in pt_peaks.items():
    theta_rad = np.radians(info['2theta'] / 2)
    fwhm_rad = 0.9 * 0.15406 / (D_crystallite * np.cos(theta_rad))
    fwhm_deg = np.degrees(fwhm_rad)
    intensity = info['rel_intensity'] * 10
    peak = generate_xrd_peak(two_theta, info['2theta'], intensity, fwhm_deg)
    xrd_signal += peak
    peak_data.append({'hkl': hkl, '2theta': info['2theta'],
                      'fwhm_deg': fwhm_deg, 'intensity': intensity})

np.random.seed(42)
xrd_signal += np.random.normal(0, 3, len(two_theta))
xrd_signal = np.maximum(xrd_signal, 0)

# Plot XRD pattern
fig, ax = plt.subplots(figsize=(12, 7))
ax.plot(two_theta, xrd_signal, '-', color=COLORS['blue'], linewidth=1.2)
for pdata in peak_data:
    idx_peak = np.argmin(np.abs(two_theta - pdata['2theta']))
    ax.annotate(
        f"Pt {pdata['hkl']}\nFWHM = {pdata['fwhm_deg']:.2f}$^\\circ$",
        xy=(pdata['2theta'], xrd_signal[idx_peak]),
        xytext=(pdata['2theta'] + 5, xrd_signal[idx_peak] + 100),
        fontsize=10, color=COLORS['vermillion'],
        arrowprops=dict(arrowstyle='->', color=COLORS['vermillion'], lw=1.5),
        bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8))
ax.set_xlabel('$2\\theta$ (degrees)')
ax.set_ylabel('Intensity (counts)')
ax.set_title(f'XRD Pattern: Pt/Al$_2$O$_3$ ($D \\approx$ {D_crystallite:.0f} nm)')
ax.set_xlim(20, 80)
plt.tight_layout()
plt.show()

# Apply Scherrer equation to each peak
print("\nScherrer Analysis:")
print("=" * 55)
print(f"{'Peak':>8} | {'2theta (deg)':>12} | {'FWHM (deg)':>11} | {'D (nm)':>8}")
print("-" * 55)
for pdata in peak_data:
    D = scherrer_size(pdata['2theta'], pdata['fwhm_deg'])
    print(f"{pdata['hkl']:>8} | {pdata['2theta']:>12.2f} | {pdata['fwhm_deg']:>11.3f} | {D:>8.1f}")
print(f"\nExpected crystallite size: {D_crystallite:.1f} nm")

---
## Part 11: Metal Dispersion and Active Surface Area

**Dispersion** $D$ is the fraction of metal atoms located on the particle surface (and thus accessible for catalysis):

$$D = \frac{N_{\text{surface}}}{N_{\text{total}}}$$

For roughly spherical particles, dispersion relates to particle diameter $d_p$ via $D \approx C / d_p$ where $C$ is a metal-specific constant (in nm).

In [ ]:
def dispersion_from_size(d_p, metal='Pt'):
    """
    Estimate metal dispersion from particle diameter.

    Parameters
    ----------
    d_p : float or array-like
        Particle diameter (nm)
    metal : str, optional
        Metal identity. Supported: 'Pt', 'Pd', 'Au', 'Ni'

    Returns
    -------
    float or ndarray
        Dispersion (0 to 1)

    Notes
    -----
    Uses D ~ C/d_p where C depends on metal:
      Pt: 1.13 nm, Pd: 1.12 nm, Au: 1.17 nm, Ni: 0.97 nm
    """
    C_values = {'Pt': 1.13, 'Pd': 1.12, 'Au': 1.17, 'Ni': 0.97}
    if metal not in C_values:
        raise ValueError(f"Metal '{metal}' not supported.")
    d_p = np.asarray(d_p, dtype=float)
    D = C_values[metal] / d_p
    return np.minimum(D, 1.0)


def active_surface_area(m_cat, w_metal, D, M_metal, sigma_metal):
    """
    Calculate metal active surface area from chemisorption data.

    Parameters
    ----------
    m_cat : float
        Catalyst mass (g)
    w_metal : float
        Metal weight fraction
    D : float
        Metal dispersion (fraction)
    M_metal : float
        Atomic mass of metal (g/mol)
    sigma_metal : float
        Surface area per metal atom (m^2)

    Returns
    -------
    dict with S_metal_m2_per_g_cat, S_metal_m2_per_g_metal, N_sites_mol
    """
    m_metal = m_cat * w_metal
    n_total = m_metal / M_metal
    n_surface = n_total * D
    S_total = n_surface * N_A * sigma_metal
    return {
        'S_metal_m2_per_g_cat': S_total / m_cat,
        'S_metal_m2_per_g_metal': S_total / m_metal,
        'N_sites_mol': n_surface
    }


# Dispersion vs particle size plot
d_p_range = np.linspace(0.5, 50, 200)
fig, ax = plt.subplots(figsize=(10, 7))

metals = ['Pt', 'Pd', 'Au', 'Ni']
metal_colors = [COLORS['blue'], COLORS['orange'], COLORS['green'],
                COLORS['vermillion']]

for metal, color in zip(metals, metal_colors):
    D = dispersion_from_size(d_p_range, metal=metal)
    ax.plot(d_p_range, D * 100, '-', color=color, linewidth=2.5, label=metal)

ax.axvspan(1, 5, alpha=0.08, color=COLORS['skyblue'],
           label='Typical catalyst range')
ax.set_xlabel('Particle Diameter, $d_p$ (nm)')
ax.set_ylabel('Dispersion, $D$ (%)')
ax.set_title('Metal Dispersion vs Particle Size')
ax.set_xlim(0, 50)
ax.set_ylim(0, 105)
ax.legend(loc='upper right')
plt.tight_layout()
plt.show()

---
## Part 12: Complete Characterization Workflow

A complete catalyst characterization combines multiple techniques to build a comprehensive picture:

1. **BET** provides total surface area (support + metal)
2. **Chemisorption** (or XRD + dispersion) provides active metal surface area
3. **TPD** provides binding energies on different site types

Together, these data feed into reactor design and kinetic modeling.

In [ ]:
# Complete characterization of a 2% Pt/Al2O3 catalyst
print("=" * 65)
print("COMPLETE CATALYST CHARACTERIZATION: 2% Pt/Al2O3")
print("=" * 65)

m_cat = 1.000; w_Pt = 0.02; M_Pt = 195.08; sigma_Pt = 8.07e-20

# Step 1: BET Surface Area
print("\n1. BET SURFACE AREA (N2 physisorption, 77 K)")
print("-" * 50)
S_BET = results['S_BET']; V_m = results['V_m']; c_BET = results['c']
print(f"  V_m = {V_m:.1f} cm^3(STP)/g, c = {c_BET:.0f}")
print(f"  S_BET = {S_BET:.0f} m^2/g")

# Step 2: XRD Crystallite Size
print("\n2. XRD CRYSTALLITE SIZE (Scherrer equation)")
print("-" * 50)
D_xrd = 5.0
print(f"  Crystallite size D = {D_xrd:.1f} nm")

# Step 3: Metal Dispersion
print("\n3. METAL DISPERSION")
print("-" * 50)
D_disp = dispersion_from_size(D_xrd, metal='Pt')
print(f"  From XRD size ({D_xrd} nm): D = {D_disp:.2f} ({D_disp*100:.0f}%)")

# Step 4: Active Surface Area
print("\n4. ACTIVE (METAL) SURFACE AREA")
print("-" * 50)
asa = active_surface_area(m_cat, w_Pt, D_disp, M_Pt, sigma_Pt)
print(f"  S_Pt = {asa['S_metal_m2_per_g_cat']:.2f} m^2/g_cat")
print(f"  Surface Pt sites = {asa['N_sites_mol']:.2e} mol/g_cat")

# Step 5: TPD Binding Energies
print("\n5. TPD BINDING ENERGIES")
print("-" * 50)
print(f"  Weak sites (alpha):  E_d = {E_d_alpha/1000:.1f} kJ/mol")
print(f"  Strong sites (beta): E_d = {E_d_beta/1000:.1f} kJ/mol")

# Summary
print("\n" + "=" * 65)
print("SUMMARY")
print("=" * 65)
fraction_active = asa['S_metal_m2_per_g_cat'] / S_BET * 100
print(f"  Total surface area (BET):     {S_BET:.0f} m^2/g")
print(f"  Active surface area (Pt):     {asa['S_metal_m2_per_g_cat']:.2f} m^2/g")
print(f"  Active fraction:              {fraction_active:.2f}%")
print(f"  Pt particle size (XRD):       {D_xrd:.1f} nm")
print(f"  Pt dispersion:                {D_disp*100:.0f}%")
print(f"\n  The vast majority of surface area is the alumina support.")
print(f"  Only {fraction_active:.2f}% is active Pt surface.")

### Key Takeaways — Part II: Catalyst Characterization

- **BET** measures total surface area via N₂ physisorption; valid for $0.05 < P/P_0 < 0.30$
- **TPD** extracts desorption energies from peak temperatures (Redhead equation)
- **XRD/Scherrer** gives crystallite size from peak broadening; always correct for instrumental contribution
- **Dispersion** = fraction of metal atoms on the surface; $D \approx C/d_p$
- Multiple techniques together give a complete catalyst picture: area, sites, structure

> **Session Break (suggested):** You have completed all 12 parts. Take a break before attempting the exercises. Review the Key Takeaways above and the reflection questions at the end.

---
## Exercises

### Exercise 1: CSTR Design with LH Kinetics

A catalytic reaction follows Langmuir-Hinshelwood kinetics:
$$r = \frac{k_s K C_A}{1 + K C_A}$$

with $k_s = 0.10$ mol/(L s), $K = 8.0$ L/mol, and inlet concentration $C_{A0} = 2.0$ mol/L.

**Task:** 
1. Calculate the space time required for 90% conversion in a CSTR
2. Plot X vs tau for this system (both CSTR and PFR)
3. What is the CSTR/PFR volume ratio at 90% conversion?

In [ ]:
# YOUR CODE HERE
# 1. Use fsolve to find tau for X = 0.9 in the CSTR
#    Hint: rearrange the CSTR design equation to solve for tau given X
# 2. Use solve_ivp to get the PFR conversion profile
# 3. Calculate the volume ratio

pass  # Replace with your implementation

### Exercise 2: CSTRs in Series vs PFR

For the same LH kinetics as Exercise 1:

**Task:**
1. Compare 1, 3, 5, and 10 equal-volume CSTRs in series against a single PFR
2. How many CSTRs in series are needed to achieve within 5% of PFR performance at 90% conversion?

**Hint:** For CSTRs in series with non-first-order kinetics, solve sequentially: feed CSTR1 output into CSTR2, etc.

In [ ]:
# YOUR CODE HERE
# For LH kinetics, CSTRs in series must be solved sequentially.
# For each stage i: tau_i = C_A0_i * X_i / r(C_A_exit_i)
# where C_A0_i is the inlet to stage i (= exit of stage i-1)

pass  # Replace with your implementation

### Exercise 3: TOF Calculation from Flow Reactor Data

A 5% Pt/Al$_2$O$_3$ catalyst (Pt dispersion = 40%) is used in a flow reactor for propane dehydrogenation. The feed contains 10% C$_3$H$_8$ in He at 1 atm total pressure and 600 K. At steady state with WHSV = 2.0 h$^{-1}$ and 0.500 g of catalyst, the exit conversion is 15%.

**Task:**
1. Calculate the molar flow rate of propane ($F_{C3H8,0}$) assuming ideal gas
2. Calculate the TOF in s$^{-1}$
3. If the catalyst deactivates linearly, losing 2% of its TOF per hour, what is the TON after 100 hours?

In [ ]:
# YOUR CODE HERE
# Hint: Use ideal gas law to get molar flow from WHSV
# WHSV = (mass flow of feed) / (mass of catalyst)
# For TON with deactivation: integrate TOF(t) from 0 to 100 h

pass  # Replace with your implementation

### Exercise 4: BET Analysis with Real Data — Critical Evaluation

Load the provided dataset `data/adsorption_isotherm_sample.csv` and perform a BET analysis.

> **Note:** Not all adsorption data are suitable for BET analysis. Part of this exercise is evaluating whether the method is appropriate.

**Task:**
1. Load the data and convert units (mol/kg → cm$^3$(STP)/g: multiply by 22414/1000)
2. Plot the full adsorption isotherm. Identify its **IUPAC type** (I through VI)
3. Apply the BET transform and perform linear regression in $0.05 < P/P_0 < 0.30$
4. Extract $V_m$, $c$, and $S_{\text{BET}}$
5. **Critical evaluation:** The data is from an activated carbon (microporous material). Based on the lecture discussion of isotherm types, should BET be applied here? What does the high $c$ value suggest? What alternative method (t-plot, Langmuir) would be more appropriate?

**Hint:** To convert mol/kg to cm$^3$(STP)/g: multiply by $V_{\text{mol}}$ (22414 cm$^3$/mol) and divide by 1000 (kg to g).

In [ ]:
# YOUR CODE HERE
# 1. Load the CSV file
# 2. Convert units: V (cm^3(STP)/g) = amount (mol/kg) * 22414 / 1000
# 3. Use bet_analysis() or implement your own
# 4. Make a publication-quality BET plot

pass  # Replace with your implementation

### Exercise 5: Scherrer Analysis from XRD Data

A Pd/C catalyst shows the following XRD peaks:

| Peak | 2$\theta$ (deg) | FWHM (deg) |
|------|-----------------|------------|
| Pd(111) | 40.12 | 1.85 |
| Pd(200) | 46.66 | 2.10 |
| Pd(220) | 68.12 | 2.45 |

**Task:**
1. Apply the Scherrer equation to each peak (use $K = 0.9$, Cu K$\alpha$)
2. Correct for instrumental broadening: $\beta_{\text{inst}} = 0.10^\circ$, $\beta = \sqrt{\beta_{\text{obs}}^2 - \beta_{\text{inst}}^2}$
3. Calculate the average crystallite size
4. Estimate the Pd dispersion from this size

In [ ]:
# YOUR CODE HERE
# 1. Apply Scherrer equation to each peak
# 2. Correct for instrumental broadening before applying Scherrer
# 3. Average the crystallite sizes
# 4. Use dispersion_from_size() with metal='Pd'

pass  # Replace with your implementation

### Exercise 6: Complete Catalyst Comparison

Two catalysts are being evaluated for CO oxidation:

| Property | Catalyst A | Catalyst B |
|----------|-----------|------------|
| Metal | Pt | Pd |
| Loading | 1% | 5% |
| S_BET (m$^2$/g) | 200 | 150 |
| Particle size (nm) | 2.5 | 8.0 |
| CO-TPD T_p (K) | 450 | 380 |

**Task:**
1. Calculate the dispersion and active surface area for each catalyst
2. Calculate the CO desorption energy for each (use $\beta = 10$ K/min)
3. If Catalyst A has a rate of 0.05 mol/(g$_{\text{cat}}$ h) and B has 0.12 mol/(g$_{\text{cat}}$ h), calculate the TOF for each
4. Which catalyst is intrinsically more active? Which produces more product per gram?

In [ ]:
# YOUR CODE HERE
# 1. Use dispersion_from_size() and active_surface_area()
# 2. Use redhead_equation() for both T_p values (beta = 10 K/min = 10/60 K/s)
# 3. TOF = rate / N_sites
# 4. Compare and discuss

pass  # Replace with your implementation

---
## Reflection Questions

1. Why does the PFR always outperform the CSTR for positive-order kinetics? Under what conditions would the CSTR be preferred despite its lower conversion efficiency?

2. For Langmuir-Hinshelwood kinetics, the effective reaction order transitions from zeroth to first as concentration decreases. How does this affect the CSTR vs PFR comparison compared to pure first-order kinetics?

3. A catalyst has TOF = 10 s$^{-1}$ but TON = 1000. Another has TOF = 1 s$^{-1}$ but TON = 100,000. Which would you choose for an industrial process, and why?

4. The automotive catalytic converter achieves >95% conversion in approximately 50 ms contact time. What does this imply about the required TOF and the role of transport limitations?

5. The BET equation is only valid in the range $0.05 < P/P_0 < 0.30$. What physical phenomena cause it to break down outside this range (both at lower and higher relative pressures)?

6. The Redhead equation assumes a known pre-exponential factor $\nu \approx 10^{13}$ s$^{-1}$. If $\nu$ were actually $10^{10}$ s$^{-1}$ (a 1000-fold error), how much would the calculated $E_d$ change? Is the Redhead equation robust to this uncertainty?

7. A catalyst has very small metal nanoparticles (< 2 nm) that are not detected by XRD. How would you determine the particle size and dispersion? What complementary techniques could you use?

8. Why does a supported metal catalyst with 50% dispersion often have better performance than one with 100% dispersion (single atoms)? Consider both electronic and geometric effects.

---
## Summary

In this notebook, we covered both reactor design and catalyst characterization -- the two pillars of Chapter 3.

**Part I -- Reactor Design (Parts 1-7):**
- Solved batch, CSTR, and PFR design equations for both power-law and LH kinetics
- Compared differential and integral methods for kinetic analysis
- Predicted reactor performance across temperatures using Arrhenius $k(T)$
- Demonstrated that PFR always outperforms CSTR for positive-order kinetics
- Showed that CSTRs in series approach PFR performance
- Introduced TOF/TON as site-normalized catalyst activity metrics
- Connected space velocity (WHSV, GHSV) to space time

**Part II -- Catalyst Characterization (Parts 8-12):**
- Performed BET surface area analysis from N$_2$ adsorption data
- Applied the Redhead equation to extract desorption energies from TPD spectra
- Used the Scherrer equation to estimate crystallite size from XRD peak widths
- Calculated metal dispersion and active surface area
- Combined all techniques into a complete catalyst characterization workflow

**Data files used:** `data/adsorption_isotherm_sample.csv`

**Key functions defined:** `batch_analytical`, `batch_lh_ode`, `cstr_conversion_analytical`, `cstr_lh_residual`, `pfr_conversion_analytical`, `pfr_ode`, `lh_rate_CA`, `cstr_series_conversion`, `calculate_tof`, `whsv_to_tau`, `ghsv_to_tau`, `bet_transform`, `bet_analysis`, `redhead_equation`, `asymmetric_gaussian`, `scherrer_size`, `generate_xrd_peak`, `dispersion_from_size`, `active_surface_area`

**Next notebook:** NB4 covers transport limitations and effectiveness factors (Chapter 4).